# Final Serialized Pipeline / Model

This notebook is the reproducible source of truth for the final serialized model artifact. It starts from the processed train/test files, trains the final model, evaluates the untouched 2025 holdout set, and saves the model bundle plus final results.

Inputs:
- `data/processed/X_train_processed.csv`
- `data/processed/y_train.csv`
- `data/processed/X_test_2025_processed.csv`
- `data/processed/y_test_2025.csv`
- `data/processed/player_lookup_train.csv`
- `data/processed/player_lookup_test_2025.csv`

Outputs:
- `artifacts/final_model.joblib`
- `results/final/final_metrics.csv`
- `results/final/final_predictions.csv`
- `results/final/final_model_metadata.json`

In [1]:
import json
import platform
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

processed_dir = ROOT / "data" / "processed"
artifact_dir = ROOT / "artifacts"
final_result_dir = ROOT / "results" / "final"

RANDOM_STATE = 26
TRAIN_YEARS = [2021, 2022, 2023, 2024]
TEST_YEAR = 2025
RESIDUAL_DEFINITION = "predicted - actual"

### Load Processed Train/Test Data

In [2]:
X_train = pd.read_csv(processed_dir / "X_train_processed.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["salary"]
X_test = pd.read_csv(processed_dir / "X_test_2025_processed.csv")
y_test = pd.read_csv(processed_dir / "y_test_2025.csv")["salary"]
lookup_train = pd.read_csv(processed_dir / "player_lookup_train.csv")
lookup_test = pd.read_csv(processed_dir / "player_lookup_test_2025.csv")

feature_names = X_train.columns.tolist()

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError(f"Train/test feature count mismatch: {X_train.shape[1]} vs {X_test.shape[1]}")

if X_train.columns.tolist() != X_test.columns.tolist():
    raise ValueError("Train/test feature columns are not aligned in the same order.")

if len(X_train) != len(y_train):
    raise ValueError("X_train and y_train row counts do not match.")

if len(X_test) != len(y_test) or len(X_test) != len(lookup_test):
    raise ValueError("X_test, y_test, and lookup_test row counts do not match.")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of final features:", len(feature_names))
display(pd.DataFrame({"feature": feature_names}).head(30))

X_train shape: (742, 25)
X_test shape: (223, 25)
Number of final features: 25


,feature
0,avail_rate
1,blk
2,fg
3,fg_per_g
4,fga
5,ft
6,fta
7,g
8,mp
9,pca1


### Train Final Model

The final model uses the tuned Random Forest configuration selected during optimization. This setup balances predictive performance with basic overfitting control.

The model uses 400 trees for stable ensemble predictions, `min_samples_leaf=5` to avoid overly specific leaf nodes, no fixed tree-depth limit because it performed best in tuning, and a fixed random seed (`random_state=26`) for reproducibility.

In [3]:
# Final tuned Random Forest selected by CV RMSE during optimization
final_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=5,
    max_depth=None,
    random_state=RANDOM_STATE,
)

final_model.fit(X_train, y_train)

print("Final model:", final_model)
print("Model parameters:")

# Show all model hyperparameters for auditability, including sklearn defaults
display(pd.Series(final_model.get_params()))

Final model: RandomForestRegressor(min_samples_leaf=5, n_estimators=400, random_state=26)
Model parameters:


bootstrap                            True
ccp_alpha                             0.0
criterion                   squared_error
max_depth                            None
max_features                          1.0
max_leaf_nodes                       None
max_samples                          None
min_impurity_decrease                 0.0
min_samples_leaf                        5
min_samples_split                       2
min_weight_fraction_leaf              0.0
monotonic_cst                        None
n_estimators                          400
n_jobs                               None
oob_score                           False
random_state                           26
verbose                                 0
warm_start                          False
dtype: object

### Holdout Prediction and Evaluation

This section applies the final trained model to the untouched 2025 holdout set. It creates player-level predictions with actual salary, predicted salary, residual, and absolute error, then computes the final KPI metrics.

Residuals are defined as `predicted - actual`. Positive residuals indicate the model predicted a higher salary than the player actually earned, while negative residuals indicate the model predicted a lower salary than the player actually earned.

The largest absolute errors are displayed as a quick diagnostic to identify where the model missed most.

In [4]:
# Generate holdout salary predictions for the untouched 2025 test set
pred = final_model.predict(X_test)

# Build a player-level prediction table with identifiers, actual salary, predicted salary, and error columns
pred_df = lookup_test.copy()
pred_df["actual"] = y_test.to_numpy()
pred_df["predicted"] = pred
pred_df["residual"] = pred_df["predicted"] - pred_df["actual"]
pred_df["abs_error"] = pred_df["residual"].abs()

# Keep the final prediction output columns in a clear reporting order
pred_cols = ["player", "team", "year", "group", "actual", "predicted", "residual", "abs_error"]
pred_df = pred_df[pred_cols]

# Compute final holdout KPIs and store split/model context for traceability
metrics = {
    "model": "RandomForestRegressor",
    "train_years": "2021-2024",
    "test_year": TEST_YEAR,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(len(feature_names)),
    "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
    "MAE": float(mean_absolute_error(y_test, pred)),
    "MAPE": float(mean_absolute_percentage_error(y_test, pred)),
    "R2": float(r2_score(y_test, pred)),
    "residual_definition": RESIDUAL_DEFINITION,
}

# Convert metrics to a one-row table for display and CSV export
metrics_df = pd.DataFrame([metrics])

display(metrics_df)

# Show the largest player-level prediction errors as a quick model diagnostic
display(pred_df.sort_values("abs_error", ascending=False).head(10))

,model,train_years,test_year,n_train,n_test,n_features,RMSE,MAE,MAPE,R2,residual_definition
0,RandomForestRegressor,2021-2024,2025,742,223,25,41368.597846,30376.251304,1.287013,0.634203,predicted - actual


,player,team,year,group,actual,predicted,residual,abs_error
54,Odyssey Sims,TOT,2025,veteran,7949,138897.386595,130948.386595,130948.386595
53,Odyssey Sims,TOT,2025,veteran,13911,138897.386595,124986.386595,124986.386595
194,Moriah Jefferson,CHI,2025,veteran,145500,22577.850691,-122922.149309,122922.149309
9,Sabrina Ionescu,NYL,2025,rookie,222060,106060.957794,-115999.042206,115999.042206
112,Teaira McCowan,DAL,2025,veteran,201400,89401.316216,-111998.683784,111998.683784
16,Arike Ogunbowale,DAL,2025,unknown,249032,139530.215438,-109501.784562,109501.784562
44,Jewell Loyd,LVA,2025,veteran,249032,140885.526888,-108146.473112,108146.473112
103,Kaila Charles,TOT,2025,veteran,13911,113460.868377,99549.868377,99549.868377
104,Kaila Charles,TOT,2025,veteran,13911,113460.868377,99549.868377,99549.868377
91,Haley Jones,TOT,2025,veteran,36094,128767.144015,92673.144015,92673.144015


### Save Final Results

In [5]:
# Save the final KPI table for reporting and reproducibility
metrics_path = final_result_dir / "final_metrics.csv"
# Save player-level final predictions for error analysis and plots
predictions_path = final_result_dir / "final_predictions.csv"

metrics_df.to_csv(metrics_path, index=False)
pred_df.to_csv(predictions_path, index=False)

### Save Final Model Bundle

This section saves the final trained model as a reusable `joblib` bundle and saves a separate JSON metadata file for easy inspection.

The `final_model.joblib` file is used to reload the trained model in Python. It includes the model, feature names, model parameters, metrics, split information, target name, residual definition, and metadata.

The `final_model_metadata.json` file is a human-readable model summary. It records the model type, selected features, train/test split, final KPI results, input/output files, and package versions.

In [6]:
# Store the chronological train/test split details used to build the final model
split_info = {
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
}

# Build metadata for the JSON file and also include it inside the joblib bundle
metadata = {
    "artifact_name": "final_model.joblib",
    "model_type": type(final_model).__name__,
    "model_params": final_model.get_params(),
    "feature_names": feature_names,
    "target": "salary",
    "split_info": split_info,
    "metrics": metrics,
    "residual_definition": RESIDUAL_DEFINITION,
    "environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
        "joblib_version": joblib.__version__,
    },
}

# Joblib bundle stores the trained model and everything needed to reuse it safely
model_bundle = {
    "model": final_model,
    "feature_names": feature_names,
    "model_params": final_model.get_params(),
    "metrics": metrics,
    "split_info": split_info,
    "target": "salary",
    "residual_definition": RESIDUAL_DEFINITION,
    "metadata": metadata,
}

model_path = artifact_dir / "final_model.joblib"
metadata_path = final_result_dir / "final_model_metadata.json"

joblib.dump(model_bundle, model_path)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)